# Test Notebook - Part 3

### Test các hàm sau: 
- solve_via_gauss
- solve_via_cholesky
- solve_via_normal_equations
- solve_gauss_seidel
- nhom benchmark/stability

Moi nhom ham chinh co 5 test case.

In [1]:
import os
import sys
import unittest

import numpy as np

THIS_DIR = os.getcwd()
if THIS_DIR not in sys.path:
    sys.path.append(THIS_DIR)

import benchmark as bm
from solvers import (
    solve_via_gauss,
    solve_via_cholesky,
    solve_via_normal_equations,
    solve_gauss_seidel,
)

In [2]:
class TestSolveViaGaussPart3(unittest.TestCase):
    def test_case_1_square_unique(self):
        A = [[2.0, 1.0], [1.0, 3.0]]
        b = [5.0, 6.0]
        x = solve_via_gauss(A, b)
        self.assertTrue(np.allclose(np.matmul(A, x), b, atol=1e-8))

    def test_case_2_requires_pivot(self):
        A = [[0.0, 2.0], [3.0, 4.0]]
        b = [4.0, 11.0]
        x = solve_via_gauss(A, b)
        self.assertTrue(np.allclose(np.matmul(A, x), b, atol=1e-8))

    def test_case_3_zero_rhs(self):
        A = [[3.0, 1.0], [1.0, 2.0]]
        b = [0.0, 0.0]
        x = solve_via_gauss(A, b)
        self.assertTrue(np.allclose(x, [0.0, 0.0], atol=1e-8))

    def test_case_4_singular_raises(self):
        A = [[1.0, 1.0], [2.0, 2.0]]
        b = [1.0, 2.0]
        with self.assertRaises(ValueError):
            solve_via_gauss(A, b)

    def test_case_5_infinite_solution_raises(self):
        A = [[1.0, 1.0], [2.0, 2.0]]
        b = [3.0, 6.0]
        with self.assertRaises(ValueError):
            solve_via_gauss(A, b)

In [3]:
class TestSolveViaCholeskyPart3(unittest.TestCase):
    def test_case_1_spd_system(self):
        A = [[4.0, 12.0, -16.0], [12.0, 37.0, -43.0], [-16.0, -43.0, 98.0]]
        b = [1.0, 2.0, 3.0]
        x = solve_via_cholesky(A, b)
        self.assertTrue(np.allclose(np.matmul(A, x), b, atol=1e-7))

    def test_case_2_diagonal_spd(self):
        A = [[2.0, 0.0], [0.0, 8.0]]
        b = [4.0, 16.0]
        x = solve_via_cholesky(A, b)
        self.assertTrue(np.allclose(x, [2.0, 2.0], atol=1e-9))

    def test_case_3_zero_rhs(self):
        A = [[9.0, 0.0], [0.0, 4.0]]
        b = [0.0, 0.0]
        x = solve_via_cholesky(A, b)
        self.assertTrue(np.allclose(x, [0.0, 0.0], atol=1e-9))

    def test_case_4_non_spd_raises(self):
        A = [[1.0, 2.0], [2.0, 1.0]]
        b = [1.0, 1.0]
        with self.assertRaises(ValueError):
            solve_via_cholesky(A, b)

    def test_case_5_mismatch_b_raises(self):
        A = [[2.0, 0.0], [0.0, 3.0]]
        b = [1.0]
        with self.assertRaises(ValueError):
            solve_via_cholesky(A, b)

In [4]:
class TestSolveViaNormalEqPart3(unittest.TestCase):
    def test_case_1_square_exact(self):
        A = [[2.0, 1.0], [1.0, 2.0]]
        b = [3.0, 3.0]
        x = solve_via_normal_equations(A, b)
        self.assertTrue(np.allclose(np.matmul(A, x), b, atol=1e-7))

    def test_case_2_rectangular_least_squares(self):
        A = [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]
        b = [1.0, 2.0, 3.0]
        x = solve_via_normal_equations(A, b)
        lhs = np.matmul(np.matmul(np.transpose(A), A), x)
        rhs = np.matmul(np.transpose(A), b)
        self.assertTrue(np.allclose(lhs, rhs, atol=1e-6))

    def test_case_3_zero_rhs(self):
        A = [[1.0, 0.0], [0.0, 2.0]]
        b = [0.0, 0.0]
        x = solve_via_normal_equations(A, b)
        self.assertTrue(np.allclose(x, [0.0, 0.0], atol=1e-9))

    def test_case_4_dependent_columns_raises(self):
        A = [[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]]
        b = [1.0, 2.0, 3.0]
        with self.assertRaises(ValueError):
            solve_via_normal_equations(A, b)

    def test_case_5_mismatch_b_raises(self):
        A = [[1.0, 2.0], [3.0, 4.0]]
        b = [1.0]
        with self.assertRaises(ValueError):
            solve_via_normal_equations(A, b)

In [5]:
class TestGaussSeidelPart3(unittest.TestCase):
    def test_case_1_diagonally_dominant_converges(self):
        A = [[4.0, 1.0], [2.0, 3.0]]
        b = [1.0, 2.0]
        x = solve_gauss_seidel(A, b, tolerance=1e-10, max_iterations=500, verbose=False)
        self.assertTrue(np.allclose(np.matmul(A, x), b, atol=1e-6))

    def test_case_2_strict_mode_non_dominant_raises(self):
        A = [[1.0, 2.0], [2.0, 1.0]]
        b = [1.0, 1.0]
        with self.assertRaises(ValueError):
            solve_gauss_seidel(A, b, strict_convergence=True, verbose=False)

    def test_case_3_zero_diagonal_raises(self):
        A = [[0.0, 1.0], [1.0, 2.0]]
        b = [1.0, 2.0]
        with self.assertRaises(ValueError):
            solve_gauss_seidel(A, b, verbose=False)

    def test_case_4_bad_x0_length_raises(self):
        A = [[4.0, 1.0], [2.0, 3.0]]
        b = [1.0, 2.0]
        with self.assertRaises(ValueError):
            solve_gauss_seidel(A, b, x0=[0.0], verbose=False)

    def test_case_5_max_iteration_returns_vector(self):
        A = [[4.0, 1.0], [2.0, 3.0]]
        b = [1.0, 2.0]
        x = solve_gauss_seidel(A, b, max_iterations=1, tolerance=1e-20, verbose=False)
        self.assertEqual(len(x), 2)

In [6]:
class TestBenchmarkAndStabilityPart3(unittest.TestCase):
    def test_case_1_run_benchmark_has_expected_methods(self):
        old_builder = bm._build_cases
        try:
            def fake_builder(n_list, num_runs):
                cases = {}
                for n in n_list:
                    A = [[4.0, 1.0], [2.0, 3.0]]
                    b = [1.0, 2.0]
                    cases[n] = [(A, b) for _ in range(num_runs)]
                return cases
            bm._build_cases = fake_builder
            out = bm.run_benchmark(verbose=False)
            self.assertIn('Khu Gauss', out)
            self.assertIn('Phan ra Cholesky', out)
            self.assertIn('He PT Chuan (Cholesky)', out)
            self.assertIn('Lap Gauss-Seidel', out)
        finally:
            bm._build_cases = old_builder

    def test_case_2_run_benchmark_contains_all_sizes(self):
        old_builder = bm._build_cases
        try:
            def fake_builder(n_list, num_runs):
                cases = {}
                for n in n_list:
                    A = [[4.0, 1.0], [2.0, 3.0]]
                    b = [1.0, 2.0]
                    cases[n] = [(A, b) for _ in range(num_runs)]
                return cases
            bm._build_cases = fake_builder
            out = bm.run_benchmark(verbose=False)
            for n in [50, 100, 200, 500, 1000]:
                self.assertIn(n, out['Khu Gauss'])
        finally:
            bm._build_cases = old_builder

    def test_case_3_run_benchmark_stability_structure(self):
        out = bm.run_benchmark_stability(verbose=False, n=5, num_runs=1)
        self.assertIn('SPD', out)
        self.assertIn('Hilbert', out)
        self.assertIn('Khu Gauss', out['SPD'])
        self.assertIn('Lap Gauss-Seidel', out['Hilbert'])

    def test_case_4_run_benchmark_stability_metrics_shape(self):
        out = bm.run_benchmark_stability(verbose=False, n=5, num_runs=1)
        metric = out['SPD']['Khu Gauss']
        self.assertIn('avg_time', metric)
        self.assertIn('avg_error', metric)
        self.assertIn('success_runs', metric)

    def test_case_5_run_benchmark_stability_multiple_n(self):
        out_n5 = bm.run_benchmark_stability(verbose=False, n=5, num_runs=1)
        out_n8 = bm.run_benchmark_stability(verbose=False, n=8, num_runs=1)
        self.assertIn('SPD', out_n5)
        self.assertIn('SPD', out_n8)
        self.assertIsInstance(out_n5['SPD']['Khu Gauss']['success_runs'], int)
        self.assertIsInstance(out_n8['Hilbert']['Lap Gauss-Seidel']['success_runs'], int)

In [7]:
suite = unittest.TestSuite()
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestSolveViaGaussPart3))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestSolveViaCholeskyPart3))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestSolveViaNormalEqPart3))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestGaussSeidelPart3))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestBenchmarkAndStabilityPart3))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print('--- SUMMARY ---')
print('run =', result.testsRun)
print('failures =', len(result.failures))
print('errors =', len(result.errors))

test_case_1_square_unique (__main__.TestSolveViaGaussPart3.test_case_1_square_unique) ... ok
test_case_2_requires_pivot (__main__.TestSolveViaGaussPart3.test_case_2_requires_pivot) ... ok
test_case_3_zero_rhs (__main__.TestSolveViaGaussPart3.test_case_3_zero_rhs) ... ok
test_case_4_singular_raises (__main__.TestSolveViaGaussPart3.test_case_4_singular_raises) ... ok
test_case_5_infinite_solution_raises (__main__.TestSolveViaGaussPart3.test_case_5_infinite_solution_raises) ... ok
test_case_1_spd_system (__main__.TestSolveViaCholeskyPart3.test_case_1_spd_system) ... ok
test_case_2_diagonal_spd (__main__.TestSolveViaCholeskyPart3.test_case_2_diagonal_spd) ... ok
test_case_3_zero_rhs (__main__.TestSolveViaCholeskyPart3.test_case_3_zero_rhs) ... ok
test_case_4_non_spd_raises (__main__.TestSolveViaCholeskyPart3.test_case_4_non_spd_raises) ... ok
test_case_5_mismatch_b_raises (__main__.TestSolveViaCholeskyPart3.test_case_5_mismatch_b_raises) ... ok
test_case_1_square_exact (__main__.TestSolveV

--- SUMMARY ---
run = 25
failures = 0
errors = 0
